# Diffusion CLN: a uniformly heated disk

For `-div(k grad T) + rho*c*dT/dt = f F(t)`, this example imposes zero temperature on the outer disk boundary. The source is normalized to unit integral, so `F` is total injected power and the output is the source-weighted mean temperature.

This is a **current-source RC ladder**: resistance goes to the reference, while capacitors connect successive nodes. It is not the usual thermal circuit with capacitors to ground. `length=1` represents a unit-depth extrusion. For depth `d`, resistance divides by `d` and capacity multiplies by `d`; a 3-D mesh already includes its volume, so use `length=1`.

Install `pip install -e ".[solver,test]"` first. The checks below use the closed-form disk elements and an independently derived Bessel transfer function.

In [1]:
from pathlib import Path
import sys
# Run from the repository root or docs/, or install CLN-mcp first.
for root in (Path.cwd(), Path.cwd().parent):
    if (root / "src" / "cln").is_dir():
        sys.path.insert(0, str(root / "src"))
        break
import numpy as np
from netgen.occ import Circle, OCCGeometry
from ngsolve import Mesh, CF
from scipy.special import iv
from cln.diffusion import DiffusionCLN, rc_ladder_impedance
radius, k, rhoc = 0.05, 500.0, 8960.0*385.0
geo = Circle((0, 0), radius).Face()
geo.edges.name = "outer"
mesh = Mesh(OCCGeometry(geo, dim=2).GenerateMesh(maxh=radius/12)).Curve(4)
result = DiffusionCLN(mesh, k, rhoc, CF(1), dirichlet="outer", order=6).run(nstages=4)
R, C = result["R"], result["C"]
for n in range(4):
    r_exact = 1/(8*(n+1)**5*np.pi*k)
    c_exact = (n+1)**2*(n+2)**2*np.pi*radius**2*rhoc/(2*n+3)
    assert np.isclose(R[n], r_exact, rtol=1e-6, atol=0)
    assert np.isclose(C[n], c_exact, rtol=1e-6, atol=0)
print("R [K/W]:", R)
print("C [J/K]:", C)


R [K/W]: [7.957747154594637e-05, 2.486795985806623e-06, 3.274793063644071e-07, 7.771236752015549e-08]
C [J/K]: [36124.12672737302, 195070.28432031337, 557343.6665302013, 1204137.3482867538]


In [2]:
def exact_disk(s):
    g = np.sqrt(s*rhoc/k)
    area = np.pi*radius**2
    return (area - 2*np.pi*radius*iv(1, g*radius)/(g*iv(0, g*radius)))/(area**2*rhoc*s)

assert rc_ladder_impedance(R, C, 0) == R[0]
tau = rhoc*radius**2/k
for x, tolerance in ((0.1, 1e-8), (1., 1e-6), (10., 1e-4)):
    s = x/tau
    error = abs(rc_ladder_impedance(R, C, s)/exact_disk(s)-1)
    assert error < tolerance
    print(f"s*tau={x:g}: relative error={error:.3e}")


s*tau=0.1: relative error=5.957e-13
s*tau=1: relative error=5.080e-12
s*tau=10: relative error=1.952e-08


`rc_ladder_impedance` truncates after the final resistance, so the final stored capacity is not used by that truncation. More stages generally improve the approximation near the expansion point, but finite-element error and loss of orthogonality still need checking.

For symbolic slab, cylinder, and sphere derivations, use the MCP tool `cln_diffusion_derive` with Mathematica installed. Its canonical package is shipped at `cln_mcp_server/resources/DiffusionCLN.wl`.